# 05: Volume reconstruction and H&E-cell state inference

Read a trained reconstruction model and anchor sections, predict molecular states across physical depth, and save the reconstructed volume.

## 1. Read the input data

Edit the real input paths and method parameters below. Run the cells in order using the `spagc` kernel. Each notebook reads its own inputs; it does not execute another notebook.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks/03_3d").is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Start Jupyter inside the spaGC repository.")
    ROOT = ROOT.parent


if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_ID = os.environ.get("SPAGC_RUN_ID", "00029_native_v2")
OUT = ROOT / "outputs/03_3d" / RUN_ID
DEVICE = os.environ.get(
    "SPAGC_DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu"
)
SEED = 29
MODE = "path"
USE_CELLTYPE = True
LABEL_KEY = "cell_class"
SPATIAL_KEY = "spatial_st_corrected"
Z_KEY = "z_um"
FRAME = "00029__g0_registered"
if not RUN_ID or Path(RUN_ID).name != RUN_ID or RUN_ID in {".", ".."}:
    raise ValueError("SPAGC_RUN_ID must be a safe directory name")
DATA = ROOT / "data/3d/00029_g0_native"
ANCHORS = DATA / "st/anchors_celltype_domain_niche"
CELLTYPE_LABELS = DATA / "cell_type_labels.csv"
ARTIFACTS = ROOT / "outputs/03_3d" / RUN_ID
MODEL_DIR = ARTIFACTS / "model"
VOLUME = ARTIFACTS / "reconstructed_volume.h5ad"
HE_CELLS = DATA / "derived/nuclei_registered.parquet"
if MODE not in ("baseline", "uot", "path"):
    raise ValueError("Unsupported run mode or model mode.")
print({"model_mode": MODE, "output": str(OUT)})
SECTION_RANGE = None
ROI_UM = None
MAX_CELLS_PER_ANCHOR = None
CHECKPOINT_NAME = None
THICKNESS_UM = 10.0
ODE_STEPS = 100
CHUNK_SIZE = 2048
RUN_HE_CELL_INFERENCE = False
DATA_RELEASE = "00029_g0_reproducibility_bundle_v1"
USE_NUCLEUS_PATH = False
if USE_NUCLEUS_PATH and MODE != "path":
    raise ValueError(
        "The optional supplied-nucleus experiment requires full mode and morphology paths."
    )
MAX_HE_QUERY_CELLS = None
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
from spagc._io import require_new
from spagc.reconstruction.core import DeepSpatial
from spagc.reconstruction.inputs import load_anchors

Read the measured anchors and select exactly one declared checkpoint from the preceding training step.

Read measured anchor sections, retain their expression and physical coordinates, and apply only the explicitly selected spatial or cell subset.

In [ ]:
anchors = load_anchors(
    ANCHORS,
    require_labels=USE_CELLTYPE,
    labels_path=CELLTYPE_LABELS,
    section_range=SECTION_RANGE,
    roi_um=ROI_UM,
    max_cells_per_anchor=MAX_CELLS_PER_ANCHOR,
    seed=SEED,
    label_key=LABEL_KEY,
    spatial_key=SPATIAL_KEY,
    z_key=Z_KEY,
)
candidates = sorted(MODEL_DIR.glob("*.ckpt"))
if CHECKPOINT_NAME is not None:
    checkpoint = MODEL_DIR / CHECKPOINT_NAME
elif len(candidates) == 1:
    checkpoint = candidates[0]
else:
    raise FileNotFoundError(
        "Set CHECKPOINT_NAME explicitly or provide exactly one checkpoint in MODEL_DIR."
    )

## 2. Restore and check the reconstruction model

Strictly restore the checkpoint and verify cell-type, morphology-pairing, path and optional nucleus settings against the selected run.

Specify which morphology-guided pairing and spatial-path terms enter the reconstruction.

In [ ]:
restored = DeepSpatial()
restored.load_checkpoint(str(checkpoint), sampling_method="dopri5")
if (
    restored.use_celltype != USE_CELLTYPE
    or restored.histology_config.use_morphology_path != (MODE == "path")
    or restored.histology_config.use_morphology_uot != (MODE != "baseline")
    or (restored.histology_config.use_nucleus_path != USE_NUCLEUS_PATH)
):
    raise ValueError(
        "Selected checkpoint does not match the declared model/cell-type mode."
    )

## 3. Reconstruct molecular states between anchor sections

Integrate the trained flow at physical depths between anchors. Retain endpoint-inherited annotations separately from the fresh niche/domain predictions made in the next notebook.

In [ ]:
volume = restored.reconstruct_full_volume(
    anchors,
    thickness=THICKNESS_UM,
    steps=ODE_STEPS,
    chunk_size=CHUNK_SIZE,
    device=DEVICE,
    cache_celltype_trajectory=restored.use_celltype,
)
assert (
    volume.n_obs > 0
    and np.isfinite(volume.obsm[SPATIAL_KEY]).all()
    and np.isfinite(volume.obs[Z_KEY]).all()
)
assert volume.var_names.equals(anchors[0].var_names)
for label in ["pred_domain", "pred_niche", "pred_domain_conf", "pred_niche_conf"]:
    if label in volume.obs:
        volume.obs["endpoint_" + label] = volume.obs[label].copy()
volume.uns["spagc_reconstruction"] = {
    "data_release": DATA_RELEASE,
    "coordinate_frame": FRAME,
    "mode": MODE,
}
volume_path = VOLUME
print(volume)

## 4. Infer states at supplied H&E nucleus locations

If enabled, read the supplied registered nucleus centroids and assign molecular states from propagated anchor trajectories at matching depths. These are predictions, not measured expression or one-to-one lineages.

In [ ]:
def load_registered_he_cells(path, *, max_cells=None):
    """Use supplied registered nucleus centroids as inference queries, not targets."""
    table = pd.read_parquet(path)
    if set(table.coordinate_frame) != {FRAME} or table.residual_applied_to_he.any():
        raise ValueError("H&E nuclei must remain in the unchanged registered frame.")
    if SECTION_RANGE is not None:
        table = table[table.section_id.between(*SECTION_RANGE)]
    if ROI_UM is not None:
        (xmin, ymin, xmax, ymax) = ROI_UM
        table = table[
            table.x_registered_um.between(xmin, xmax)
            & table.y_registered_um.between(ymin, ymax)
        ]
    if max_cells is not None and len(table) > max_cells:
        table = table.sample(n=int(max_cells), random_state=SEED).sort_index()
    if table.empty or table.cell_id.duplicated().any():
        raise ValueError("H&E centroid selection must be nonempty and have unique IDs.")
    table = table.set_index("cell_id")
    result = ad.AnnData(obs=table.copy())
    result.obsm[SPATIAL_KEY] = table[["x_registered_um", "y_registered_um"]].to_numpy(
        float
    )
    result.uns["query_semantics"] = (
        "supplied registered nucleus centroids; not measured expression"
    )
    return result

In [ ]:
if RUN_HE_CELL_INFERENCE:
    if not HE_CELLS.is_file():
        raise FileNotFoundError(
            f"Supply already registered H&E-cell centroids: {HE_CELLS}"
        )
    he_cells = load_registered_he_cells(HE_CELLS, max_cells=MAX_HE_QUERY_CELLS)
    pieces = []
    assigned = np.zeros(he_cells.n_obs, bool)
    for i, (a, b) in enumerate(zip(anchors[:-1], anchors[1:])):
        (low, high) = (float(a.obs[Z_KEY].iloc[0]), float(b.obs[Z_KEY].iloc[0]))
        keep = (np.asarray(he_cells.obs[Z_KEY]) >= low) & (
            np.asarray(he_cells.obs[Z_KEY]) <= high
            if i == len(anchors) - 2
            else np.asarray(he_cells.obs[Z_KEY]) < high
        )
        if keep.any():
            query = he_cells[keep].copy()
            pieces.append(
                restored.predict_on_he_cells(
                    query, a, b, steps=ODE_STEPS, chunk_size=CHUNK_SIZE, device=DEVICE
                )
            )
            assigned |= keep
    if not assigned.all():
        raise ValueError("Some H&E query depths lie outside the supplied anchor range.")
    predicted = ad.concat(pieces, uns_merge="first")
    assert predicted.n_obs == he_cells.n_obs
    predicted.write_h5ad(require_new(OUT / "he_cell_states.h5ad"), compression="gzip")

## 5. Save the reconstructed volume

Save physical coordinates, predicted molecular states and reconstruction metadata without overwriting an earlier volume.

In [ ]:
volume.write_h5ad(require_new(volume_path), compression="gzip")
print(volume_path)